In [0]:
# -------------------------------------------------------------------------------------------
# NOTEBOOK: 02_ingest_bronze_raw
# ROLE: Free-Tier Optimized Incremental Data Ingestion (Bronze Layer)
# -------------------------------------------------------------------------------------------
# 
# ⚠️  FREE TIER OPTIMIZATION NOTICE:
# This code is optimized for Databricks Community Edition (free tier) to minimize DBU usage.
# It uses sequential processing, small batch sizes, and pauses between streams.
#
# 📊 FOR PRODUCTION/PAID DATABRICKS EDITIONS:
# To optimize performance when porting to a regular Databricks workspace:
#   1. Enable parallel execution: Use ThreadPoolExecutor with max_workers=3
#   2. Increase batch sizes: MAX_FILES_PER_TRIGGER=50, MAX_BYTES_PER_TRIGGER="50mb"
#   3. Remove pauses: Set PAUSE_BETWEEN_STREAMS=0 or remove the sleep() calls
#   4. Faster polling: Adjust trigger interval for more frequent processing
#   5. Consider continuous streaming: Replace trigger(availableNow=True) with 
#      trigger(processingTime="30 seconds") for real-time ingestion
#
# 🔄 FOR CONTINUOUS FILE MONITORING (Production Real-Time Ingestion):
# Current behavior: Runs once, processes available files, then STOPS.
# To make it continuously look for new files on an ongoing basis:
#   
#   A. Change the trigger mode in ingest_stream():
#      FROM: .trigger(availableNow=True)
#      TO:   .trigger(processingTime="30 seconds")  # or "1 minute", "5 minutes", etc.
#   
#   B. Remove the query.awaitTermination() limit or run in a job/workflow
#   
#   C. This keeps the cluster running continuously, checking for new files every interval
#   
#   D. Consider scheduling this as a Databricks Job with:
#      - Always-on cluster OR
#      - Job cluster with long timeout OR  
#      - Delta Live Tables (DLT) pipeline for fully managed streaming
#
# ⚠️  WARNING: Continuous streaming incurs ongoing compute costs. Only use in production
#              when real-time ingestion is required. For batch processing, keep availableNow
#              and schedule the notebook to run periodically (e.g., hourly, daily).
# -------------------------------------------------------------------------------------------

import time
import re
import hashlib
from pyspark.sql import DataFrame
from pyspark.sql.functions import col, input_file_name
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, LongType, BinaryType
from typing import Dict, Optional

# -------------------------------------------------------------------------------------------
# 1. CONFIGURATION (FREE TIER OPTIMIZED)
# -------------------------------------------------------------------------------------------
CATALOG_NAME = "hc_regulatory_sandbox"
SCHEMA_NAME = "bronze_ingestion" 
VOLUME_ROOT = f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/landing_zone"
CHECKPOINT_BASE = f"{VOLUME_ROOT}/_checkpoints"

# Resource throttling (minimized for free tier DBU limits)
MAX_FILES_PER_TRIGGER = 10       # Small batch to avoid DBU spikes
MAX_BYTES_PER_TRIGGER = "10mb"   # Conservative limit for free tier
PAUSE_BETWEEN_STREAMS = 2        # Seconds to pause between streams (reduces sustained usage)
MAX_COLUMN_NAME_LENGTH = 240     # Unity Catalog limit is 255, leaving room for suffixes

# Schema for binaryFile format (required for streaming)
BINARY_FILE_SCHEMA = StructType([
    StructField("path", StringType(), True),
    StructField("modificationTime", TimestampType(), True),
    StructField("length", LongType(), True),
    StructField("content", BinaryType(), True)
])

print(f"🟢 FREE TIER MODE: Sequential processing with minimal resource usage")
print(f"Starting bronze ingestion at {time.strftime('%Y-%m-%d %H:%M:%S')}")

# -------------------------------------------------------------------------------------------
# 2. HELPER FUNCTIONS
# -------------------------------------------------------------------------------------------

# Pre-compile regex for performance
COLUMN_CLEAN_PATTERN = re.compile(r'[^a-zA-Z0-9_]+')

def clean_column_names(df: DataFrame) -> DataFrame:
    """Replace spaces and special characters with underscores for Delta compatibility.
    Handles empty/unnamed columns and truncates long names (Unity Catalog 255 char limit)."""
    new_names = []
    unnamed_counter = 1
    
    for column in df.columns:
        # Clean the column name
        clean_name = COLUMN_CLEAN_PATTERN.sub('_', column).strip('_')
        
        # Handle empty or blank column names (e.g., from trailing commas in CSV)
        if not clean_name:
            clean_name = f"unnamed_column_{unnamed_counter}"
            unnamed_counter += 1
        
        # Truncate if too long (Unity Catalog limit is 255 chars)
        if len(clean_name) > MAX_COLUMN_NAME_LENGTH:
            # Create a hash of the full name for uniqueness
            name_hash = hashlib.md5(clean_name.encode()).hexdigest()[:8]
            clean_name = f"{clean_name[:MAX_COLUMN_NAME_LENGTH]}_{name_hash}"
        
        # Handle duplicates by appending a counter
        original_clean_name = clean_name
        counter = 1
        while clean_name in new_names:
            # Ensure the suffix doesn't push us over the limit
            suffix = f"_{counter}"
            max_base_len = 255 - len(suffix)
            clean_name = f"{original_clean_name[:max_base_len]}{suffix}"
            counter += 1
        
        new_names.append(clean_name)
    
    # Rename all columns at once
    for old_name, new_name in zip(df.columns, new_names):
        if old_name != new_name:
            df = df.withColumnRenamed(old_name, new_name)
    
    return df

def ingest_stream(
    source_path: str, 
    table_name: str, 
    file_format: str = "csv", 
    separator: str = ",",
    schema_hints: Optional[str] = None
) -> Dict[str, any]:
    """Resource-optimized streaming ingestion with error handling and metrics
    
    Supported formats:
    - csv: Structured CSV/TSV files with Auto Loader schema inference
    - binaryFile: Raw file ingestion using Spark's native binaryFile reader (for complex XML)
    """
    full_table_name = f"{CATALOG_NAME}.{SCHEMA_NAME}.{table_name}"
    checkpoint_path = f"{CHECKPOINT_BASE}/{table_name}/offsets"
    schema_path = f"{CHECKPOINT_BASE}/{table_name}/schema"
    
    start_time = time.time()
    metrics = {"table": table_name, "status": "failed", "duration": 0, "error": None}
    
    try:
        print(f"\n[START] {table_name}")
        print(f"  Source: {source_path}")
        print(f"  Target: {full_table_name}")
        print(f"  Format: {file_format}")
        
        # Pre-check: Verify source directory exists and has files
        try:
            files = dbutils.fs.ls(source_path)
            if file_format == "binaryFile":
                data_files = [f for f in files if f.name.endswith('.xml')]
            else:
                data_files = [f for f in files if f.name.endswith(('.csv', '.txt'))]
            
            if len(data_files) == 0:
                raise ValueError(f"No data files found in {source_path}. Run data preparation notebook first.")
            
            print(f"  Found {len(data_files)} file(s) to process")
            
        except Exception as e:
            if "No such file or directory" in str(e) or "does not exist" in str(e):
                raise ValueError(f"Source directory does not exist: {source_path}. Run data preparation notebook (01b_raw_data_preparation) first.")
            elif "No data files found" in str(e):
                raise  # Re-raise our custom error
            else:
                raise ValueError(f"Error accessing {source_path}: {str(e)}")
        
        # APPROACH 1: For CSV files - use Auto Loader with schema inference
        if file_format == "csv":
            # Build stream reader with free-tier optimized Auto Loader settings
            reader = (spark.readStream
                .format("cloudFiles")
                .option("cloudFiles.format", file_format)
                .option("cloudFiles.schemaLocation", schema_path)
                .option("cloudFiles.inferColumnTypes", "true")
                .option("cloudFiles.maxFilesPerTrigger", MAX_FILES_PER_TRIGGER)
                .option("cloudFiles.useNotifications", "false")  # Disable for simplicity
                .option("sep", separator)
                .option("header", "true")
                .option("ignoreLeadingWhiteSpace", "true")
                .option("ignoreTrailingWhiteSpace", "true")
                .option("multiLine", "false"))  # Optimize for single-line records
            
            # Add schema hints if provided (reduces inference overhead)
            if schema_hints:
                reader = reader.option("cloudFiles.schemaHints", schema_hints)
            
            stream_df = reader.load(source_path)
            stream_df = clean_column_names(stream_df)
        
        # APPROACH 2: For complex XML files - use native Spark binaryFile format
        # Stores each file as a row with: path, modificationTime, length, content (binary)
        # NO PARSING - just raw content storage for downstream processing
        elif file_format == "binaryFile":
            print(f"  📄 Using native binaryFile format (raw content, no parsing)")
            # Use Spark's native binaryFile format with explicit schema (required for streaming)
            stream_df = (spark.readStream
                .format("binaryFile")  # Native Spark format, not cloudFiles
                .schema(BINARY_FILE_SCHEMA)  # Required for streaming mode
                .option("pathGlobFilter", "*.xml")
                .load(source_path))
        
        else:
            raise ValueError(f"Unsupported file format: {file_format}")
        
        # Write stream with free-tier optimized settings
        query = (stream_df.writeStream
            .format("delta")
            .option("checkpointLocation", checkpoint_path)
            .option("mergeSchema", "true")
            .option("delta.columnMapping.mode", "name")
            .option("maxBytesPerTrigger", MAX_BYTES_PER_TRIGGER)
            .trigger(availableNow=True)  # Process available data then stop (key for cost control)
            .toTable(full_table_name))
        
        query.awaitTermination()
        
        duration = time.time() - start_time
        metrics.update({
            "status": "success",
            "duration": round(duration, 2)
        })
        
        print(f"  ✅ {table_name} complete ({duration:.2f}s)")
        
    except Exception as e:
        duration = time.time() - start_time
        error_msg = str(e)
        metrics.update({
            "status": "failed",
            "duration": round(duration, 2),
            "error": error_msg
        })
        print(f"  ❌ {table_name} failed: {error_msg}")
    
    return metrics

# -------------------------------------------------------------------------------------------
# 3. EXECUTE INGESTION STREAMS (SEQUENTIAL - FREE TIER SAFE)
# -------------------------------------------------------------------------------------------

# Define ingestion jobs (OPTION A: Separate tables by category)
# XML files use binaryFile format (raw ingestion, parse in silver/gold layer)
ingestion_jobs = [
    # FDA Master Hub - Regulatory data (CSV, tab-separated)
    {
        "source_path": f"{VOLUME_ROOT}/fda_master_hub/",
        "table_name": "fda_master",
        "file_format": "csv",
        "separator": "\t"
    },
    # SPL Labels - Prescription drugs (XML ingested as binaryFile)
    {
        "source_path": f"{VOLUME_ROOT}/spl_labels/prescription/",
        "table_name": "spl_labels_prescription",
        "file_format": "binaryFile",  # Raw XML content for downstream parsing
        "separator": ","  # Not used for binaryFile
    },
    # SPL Labels - Over-the-counter drugs (XML ingested as binaryFile)
    {
        "source_path": f"{VOLUME_ROOT}/spl_labels/otc/",
        "table_name": "spl_labels_otc",
        "file_format": "binaryFile",  # Raw XML content for downstream parsing
        "separator": ","  # Not used for binaryFile
    },
    # SPL Labels - Animal drugs (XML ingested as binaryFile)
    {
        "source_path": f"{VOLUME_ROOT}/spl_labels/animal/",
        "table_name": "spl_labels_animal",
        "file_format": "binaryFile",  # Raw XML content for downstream parsing
        "separator": ","  # Not used for binaryFile
    },
    # SPL Labels - Other drugs (XML ingested as binaryFile)
    {
        "source_path": f"{VOLUME_ROOT}/spl_labels/other/",
        "table_name": "spl_labels_other",
        "file_format": "binaryFile",  # Raw XML content for downstream parsing
        "separator": ","  # Not used for binaryFile
    },
    # Provider Registry - DAC data (CSV)
    {
        "source_path": f"{VOLUME_ROOT}/provider_registry/",
        "table_name": "providers",
        "file_format": "csv",
        "separator": ","
    }
]

print(f"\n{'='*60}")
print(f"Processing {len(ingestion_jobs)} streams SEQUENTIALLY...")
print(f"Mode: FREE TIER OPTIMIZED (10 files/10MB per trigger)")
print(f"XML Strategy: Raw binary ingestion (NO parsing, downstream processing)")
print(f"{'='*60}")

# Execute streams sequentially (one at a time)
all_metrics = []
for i, job in enumerate(ingestion_jobs, 1):
    print(f"\n[{i}/{len(ingestion_jobs)}] Processing: {job['table_name']}")
    
    try:
        metrics = ingest_stream(**job)
        all_metrics.append(metrics)
        
        # Pause between streams to avoid sustained compute usage (free tier friendly)
        if i < len(ingestion_jobs):
            print(f"  ⏸️  Pausing {PAUSE_BETWEEN_STREAMS}s before next stream...")
            time.sleep(PAUSE_BETWEEN_STREAMS)
            
    except Exception as e:
        print(f"  ❌ Unexpected error in {job['table_name']}: {str(e)}")
        all_metrics.append({
            "table": job['table_name'],
            "status": "failed",
            "error": str(e)
        })

# -------------------------------------------------------------------------------------------
# 4. SUMMARY REPORT
# -------------------------------------------------------------------------------------------
print(f"\n{'='*60}")
print(f"BRONZE INGESTION SUMMARY")
print(f"{'='*60}")

successful = [m for m in all_metrics if m["status"] == "success"]
failed = [m for m in all_metrics if m["status"] == "failed"]
skipped = [m for m in failed if "does not exist" in m.get("error", "") or "No data files found" in m.get("error", "")]

print(f"✅ Successful: {len(successful)}/{len(all_metrics)}")
for m in successful:
    print(f"   • {m['table']}: {m['duration']}s")

if skipped:
    print(f"\n⏭️  Skipped (no data): {len(skipped)}/{len(all_metrics)}")
    for m in skipped:
        print(f"   • {m['table']}: {m.get('error', 'Unknown')}")

failed_other = [m for m in failed if m not in skipped]
if failed_other:
    print(f"\n❌ Failed: {len(failed_other)}/{len(all_metrics)}")
    for m in failed_other:
        print(f"   • {m['table']}: {m.get('error', 'Unknown error')}")

total_duration = sum(m.get("duration", 0) for m in all_metrics)
print(f"\nTotal duration: {total_duration:.2f}s (sequential execution)")
print(f"💡 Free tier optimized: Small batches + pauses = minimal DBU usage")

if skipped:
    print(f"\n⚠️  ACTION REQUIRED: Run notebook 01b_raw_data_preparation to extract source files")

print(f"\n📋 BRONZE LAYER SCHEMA:")
print(f"   • CSV tables: Auto Loader inferred columns + cloudFiles metadata")
print(f"   • XML tables: path, modificationTime, length, content (binary)")
print(f"   • Next step: Parse XML in silver layer using spark-xml or custom logic")

print(f"{'='*60}")